# Kiểm tra dữ liệu bằng Pandas

Notebook đọc CSV trực tiếp từ HDFS, thống kê các vấn đề dữ liệu và lưu báo cáo JSON.
Dữ liệu gốc chỉ được đọc, không bị thay đổi.

## 1. Cấu hình

In [1]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

HDFS_FILE = os.environ.get(
    "AUDIT_HDFS_FILE",
    "/data/hotel_bookings/input/hotel_bookings_synthetic_200MB.csv",
)
WEBHDFS_URL = f"http://namenode:9870/webhdfs/v1{HDFS_FILE}?op=OPEN&user.name=root"
REPORT_DIR = Path("/workspace/reports")
MISSING_WORDS = {"", "nan", "null", "none", "na", "n/a"}
ADR_LIMIT = 1000

NUMERIC_COLUMNS = [
    "is_canceled", "lead_time", "arrival_date_year",
    "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights", "adults",
    "children", "babies", "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "booking_changes", "agent", "company",
    "days_in_waiting_list", "adr", "required_car_parking_spaces",
    "total_of_special_requests",
]
NONNEGATIVE_COLUMNS = [
    "lead_time", "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights", "adults", "children",
    "babies", "previous_cancellations", "previous_bookings_not_canceled",
    "booking_changes", "days_in_waiting_list", "required_car_parking_spaces",
    "total_of_special_requests",
]
GUEST_COLUMNS = ["adults", "children", "babies"]
NIGHT_COLUMNS = ["stays_in_weekend_nights", "stays_in_week_nights"]

## 2. Đọc CSV từ HDFS

In [2]:
started_at = time.perf_counter()

df = pd.read_csv(
    WEBHDFS_URL,
    dtype="string",
    keep_default_na=False,
    low_memory=False,
)

required = set(
    NUMERIC_COLUMNS
    + ["arrival_date_month", "reservation_status", "reservation_status_date"]
)
missing_columns = sorted(required - set(df.columns))
if missing_columns:
    raise ValueError(f"CSV thiếu cột bắt buộc: {missing_columns}")

print(f"Đã đọc {len(df):,} dòng và {len(df.columns)} cột.")

Đã đọc 1,383,291 dòng và 32 cột.


## 3. Thống kê giá trị thiếu và số không hợp lệ

In [3]:
missing = {
    column: df[column].str.strip().str.lower().isin(MISSING_WORDS)
    for column in df.columns
}

missing_values = {
    column: {
        "count": int(mask.sum()),
        "percent": round(float(mask.mean() * 100), 4),
    }
    for column, mask in missing.items()
    if mask.any()
}

numbers = {}
invalid_numbers = {}
for column in NUMERIC_COLUMNS:
    numbers[column] = pd.to_numeric(
        df[column].mask(missing[column]),
        errors="coerce",
    )
    invalid = ~missing[column] & numbers[column].isna()
    if invalid.any():
        invalid_numbers[column] = int(invalid.sum())

print(f"Có {len(missing_values)} cột đang chứa giá trị thiếu.")
print(f"Có {len(invalid_numbers)} cột chứa giá trị số không hợp lệ.")

Có 4 cột đang chứa giá trị thiếu.
Có 0 cột chứa giá trị số không hợp lệ.


## 4. Kiểm tra các quy tắc dữ liệu

In [4]:
def total_is_zero(columns):
    values = pd.concat([numbers[column] for column in columns], axis=1)
    complete_rows = values.notna().all(axis=1)
    return complete_rows & values.sum(axis=1).eq(0)

zero_guests = total_is_zero(GUEST_COLUMNS)
zero_nights = total_is_zero(NIGHT_COLUMNS)
negative_adr = numbers["adr"] < 0
zero_adr = numbers["adr"] == 0
high_adr = numbers["adr"] > ADR_LIMIT

negative_values = {
    column: int((numbers[column] < 0).sum())
    for column in NONNEGATIVE_COLUMNS
    if (numbers[column] < 0).any()
}

arrival_text = (
    df["arrival_date_year"].str.strip() + "-"
    + df["arrival_date_month"].str.strip() + "-"
    + df["arrival_date_day_of_month"].str.strip()
)
arrival_date = pd.to_datetime(
    arrival_text, format="%Y-%B-%d", errors="coerce"
)
status_date = pd.to_datetime(
    df["reservation_status_date"], format="%Y-%m-%d", errors="coerce"
)

status = df["reservation_status"].str.strip()
expected_canceled = status.map({
    "Canceled": 1,
    "No-Show": 1,
    "Check-Out": 0,
})
status_mismatch = (
    numbers["is_canceled"].notna()
    & expected_canceled.notna()
    & numbers["is_canceled"].ne(expected_canceled)
)

def count_by_status(condition):
    labels = status[condition]
    labels = labels.mask(
        labels.str.lower().isin(MISSING_WORDS),
        "(missing)",
    )
    return {
        str(label): int(count)
        for label, count in labels.value_counts().sort_index().items()
    }

checks = {
    "zero_guests": {
        "count": int(zero_guests.sum()),
        "by_reservation_status": count_by_status(zero_guests),
    },
    "zero_nights": {
        "count": int(zero_nights.sum()),
        "by_reservation_status": count_by_status(zero_nights),
    },
    "negative_adr": {
        "count": int(negative_adr.sum()),
        "by_reservation_status": count_by_status(negative_adr),
    },
    "zero_adr": int(zero_adr.sum()),
    "adr_above_1000": int(high_adr.sum()),
    "negative_values_by_column": negative_values,
    "invalid_arrival_date": int(arrival_date.isna().sum()),
    "invalid_reservation_status_date": int(status_date.isna().sum()),
    "reservation_status_canceled_mismatch": int(status_mismatch.sum()),
}

## 5. Lưu báo cáo và thời gian chạy

In [5]:
duplicate_rows = int(df.duplicated().sum())
elapsed_seconds = time.perf_counter() - started_at

report = {
    "engine": "pandas",
    "source": WEBHDFS_URL,
    "read_only": True,
    "total_rows": int(len(df)),
    "total_columns": int(len(df.columns)),
    "missing_values": missing_values,
    "invalid_numeric_values": invalid_numbers,
    "exact_duplicate_rows_extra": duplicate_rows,
    "checks": checks,
    "runtime_seconds": round(elapsed_seconds, 3),
    "notes": [
        "Notebook chỉ thống kê, không sửa CSV gốc.",
        "Dòng giống hệt chưa chắc là cùng booking vì dữ liệu không có booking_id.",
        "Chỉ tính tổng khách hoặc tổng đêm bằng 0 khi mọi cột cấu thành có giá trị hợp lệ.",
        "Các chuỗi rỗng, nan, null, none, na, n/a được xem là thiếu.",
    ],
}

missing_table = pd.DataFrame(
    [{"column": column, **values} for column, values in missing_values.items()],
    columns=["column", "count", "percent"],
)
display(missing_table.sort_values("count", ascending=False))

REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "audit_pandas.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(pd.DataFrame([
    {"kiểm tra": "Dòng dữ liệu", "số lượng": report["total_rows"]},
    {"kiểm tra": "Dòng trùng hoàn toàn dư", "số lượng": report["exact_duplicate_rows_extra"]},
    {"kiểm tra": "Tổng khách bằng 0", "số lượng": checks["zero_guests"]["count"]},
    {"kiểm tra": "Tổng đêm bằng 0", "số lượng": checks["zero_nights"]["count"]},
    {"kiểm tra": "ADR âm", "số lượng": checks["negative_adr"]["count"]},
    {"kiểm tra": "Ngày đến không hợp lệ", "số lượng": checks["invalid_arrival_date"]},
]))
print(f"Thời gian đọc và kiểm tra bằng Pandas: {elapsed_seconds:.2f} giây")
print("Đã lưu báo cáo: reports/audit_pandas.json")

,column,count,percent
3,company,1304997,94.3400
2,agent,188998,13.6629
1,country,5590,0.4041
0,children,55,0.0040


,kiểm tra,số lượng
0,Dòng dữ liệu,1383291
1,Dòng trùng hoàn toàn dư,53673
2,Tổng khách bằng 0,2195
3,Tổng đêm bằng 0,8315
4,ADR âm,15
5,Ngày đến không hợp lệ,0


Thời gian đọc và kiểm tra bằng Pandas: 24.35 giây
Đã lưu báo cáo: reports/audit_pandas.json
